# PyTorch playground

An MLP (multi-layer perceptron) treats each position independently before pooling, so it mainly captures composition-level effects. A CNN applies convolutional filters across local windows and is good for motif detection. A Transformer encoder uses self-attention so each position can interact with all other positions, which makes it useful for modeling long-range dependencies.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as n

/home/katwre/miniforge3/envs/genomics_torch/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


# Simple PyTorch implementation
# Implement a small neural network in PyTorch for sequence-level 
# prediction.
#For example:
#input: one-hot encoded sequence or embedding
#output: a scalar prediction, like expression score

RNA/DNA sequence  →  predicted value

#sequence        predicted score
#--------------------------------
#AUGCAUG         0.83
#UUUCGAA         0.12
#GGGAUCU         0.55


input (4 features)
↓
Linear layer
↓
ReLU
↓
average over sequence
↓
Linear layer
↓
scalar output

In [ ]:
# Training data
X = [
"AUGCAU",
"UUUCGA",
"GGGAUC"
]
y = [
0.8,
0.2,
0.5
]

#A simple MLP or CNN is completely fine.


In [ ]:
d = """
                RNA sequence
                   "AUGC"
                     │
                     ▼
              One-hot encoding
            (seq_len × 4 matrix)

        A → [1 0 0 0]
        U → [0 1 0 0]
        G → [0 0 1 0]
        C → [0 0 0 1]

                     │
                     ▼
           ┌───────────────────┐
           │   Linear layer    │
           │      fc1          │
           │     4 → 16        │
           └───────────────────┘
                     │
                     ▼
                 ReLU
                     │
                     ▼
        Hidden representation per nucleotide
                shape: (seq_len, 16)

           [h11 h12 ... h1,16]
           [h21 h22 ... h2,16]
           [h31 h32 ... h3,16]
           [h41 h42 ... h4,16]

                     │
                     ▼
            Mean pooling over sequence
               average across rows

                shape: (16)

           [mean(h1), mean(h2), ... mean(h16)]

                     │
                     ▼
           ┌───────────────────┐
           │   Linear layer    │
           │      fc2          │
           │     16 → 1        │
           └───────────────────┘
                     │
                     ▼
              Scalar prediction

               expression score
                   e.g. 0.73

                   """

In [33]:
def one_hot_encode(inseq):
    # LETTERS = "AUCG" # hard-coded
    onehot = {"A": [1,0,0,0],
         "U": [0,1,0,0],
         "C": [0,0,1,0],
         "G": [0,0,0,1]}
    return torch.tensor([onehot[s] for s in inseq], dtype=torch.float32)


In [34]:
input_seq = "AAAUGCC"
# PyTorch works with tensors instead of lists.
# A tensor is just a multi-dimensional array of numbers.
# its designed for machine learning and GPU computation.
x = one_hot_encode(input_seq)
print(x)

tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 0., 1.],
        [0., 0., 1., 0.],
        [0., 0., 1., 0.]])


In [30]:
class SeqModel(nn.Module):
    def __init__(self):
        # it runs first nn.Module.__init__()
        # and then it rruns SeqModel.__init__()
        super().__init__() # calls the constructor of the parent class (nn.Module); 
                           # starts the engine of the neural network framework
                           # Run the initialization logic of the parent class.
                           # otehrwise model.parameters() wont do anything
        # Each of the 16 outputs is a weighted combination of A,U,G,C
        # Example neuron: feature1 = 0.3*A + 0.1*U + 0.8*G - 0.2*C
        self.fc1 = nn.Linear(4,16) 
        self.fc2 = nn.Linear(16,1)
    def forward(self, x):
        #print("forward pass")
        #print(x.shape)
        x = self.fc1(x) # 16 learned features per position
        #print("after linear layer")
        #print(x.shape)
        x = torch.relu(x) # This introduces non-linearity.
        #print("before mean pooling")
        #print(x.shape)
        #print(x)
        x = x.mean(dim=0) # We average across the sequence length over that 16 features (that is per position in a sequence)
        #print("after mean pooling")
        #print(x.shape)
        x = self.fc2(x) # scalar prediction
        return x
desc = """
AAAUGCC
   │
   ▼
(7 × 4) one-hot
   │
   ▼
Linear(4→16)
   │
   ▼
ReLU
   │
   ▼
(7 × 16) position features
   │
   ▼
Mean pooling
   │
   ▼
(16) sequence vector
   │
   ▼
Linear(16→1)
   │
   ▼
scalar prediction
"""

In [31]:

model = SeqModel()
prediction = model(x)
print(prediction)


tensor([-0.0569], grad_fn=<ViewBackward0>)


In [ ]:
model = SeqModel()
x = one_hot_encode("AAAUGCC")
prediction = model(x)
target = torch.tensor([0.8])
# Loss function
loss_fn = nn.MSELoss() # mean squared error loss, (prediction − target)²
loss = loss_fn(prediction, target)
print(loss) # how wrong the prediction is; 0.94 just means the prediction is far from the target.
# that snormal for untraind network. We will train the network to make better predictions.

tensor(0.9427, grad_fn=<MseLossBackward0>)


In [ ]:
#What interviewers expect: 
"""
sequence
↓
one-hot encoding
↓
model
↓
prediction
↓
compare to target
↓
loss
↓
update weights
"""

# Then we use backpropagation to update weights.
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer.zero_grad() # Clear previous gradients; PyTorch accumulates gradients, so we reset them each iteration.
prediction = model(x) # forward pass to get the prediction
loss = loss_fn(prediction, target) # Measure how wrong the prediction is.
loss.backward() # Compute gradients for all weights.
optimizer.step() #  Optimizer updates the weights using those gradients.



In [ ]:
#  -------- toy example with multiple sequences and targets
sequences = [
    "AAAUGCC",
    "AUGCGAA",
    "UUUGGCA"
]
targets = [
    0.8,
    0.3,
    0.6
]

# -------- prepare data --------
X = [one_hot_encode(seq) for seq in sequences]
Y = torch.tensor(targets, dtype=torch.float32)

# -------- model / loss / optimizer --------
model = SeqModel()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -------- training loop --------
for epoch in range(10): # one pass through the entire dataset
    total_loss = 0.0
    for x, y in zip(X,Y):
        optimizer.zero_grad()
        prediction = model(x) # forward pass to get the prediction
        loss = loss_fn(prediction, y) # Measure how wrong the prediction is.
        loss.backward() # Compute gradients for all weights.
        optimizer.step() #  Optimizer updates the weights using those gradients.
        total_loss += loss.item()
    avg_loss = total_loss / len(X)
    print(f"epoch={epoch}, avg_loss={avg_loss:.2f}")

# Here we use MSE loss
# Because the task predicts a scalar continuous value
# it's a regression task


epoch=0, avg_loss=0.49
epoch=1, avg_loss=0.47
epoch=2, avg_loss=0.45
epoch=3, avg_loss=0.43
epoch=4, avg_loss=0.41
epoch=5, avg_loss=0.40
epoch=6, avg_loss=0.38
epoch=7, avg_loss=0.36
epoch=8, avg_loss=0.34
epoch=9, avg_loss=0.33


In [ ]:
# If you shuffle the nucleotides in the sequence, will the model’s prediction change?
# In your current architecture, the prediction will not change much (and theoretically can become identical).
# Mean pooling removes sequence order.
# The model cannot learn motifs like: AUG, TATA, CpG
# How real sequence models fix this? :
# They use CNNs or transformers!

#An MLP with mean pooling mostly learns composition.
#A CNN can learn motifs.

#MLP: each position processed mostly independently
#CNN: small window slides across sequence and detects patterns

In [ ]:
# How could we improce this model?
# A limitation of this model is that it treats each position 
# independently before pooling. In real sequence modeling, 
# convolutional neural networks are often used because 
# they can detect motifs through convolution filters that slide across the sequence


In [ ]:
# CNN
"""
sequence
↓
one-hot
↓
Conv1D filters (motif detectors)
↓
ReLU
↓
pooling
↓
dense layer
↓
scalar prediction
"""
# even simpler to rememebr:#
"""
sequence
↓
one-hot encoding
↓
convolution filters scan across sequence
↓
ReLU
↓
pooling
↓
linear layer
↓
scalar prediction
"""
# which is:
"""
one-hot:          (7, 4)
transpose+batch:  (1, 4, 7)
Conv1d(4→16,k=3): (1, 16, 5)
ReLU:             (1, 16, 5)
global max pool:  (1, 16)
Linear(16→1):     (1, 1)
"""
# PyTorch Conv1d expects a different layout
# For nn.Conv1d, input shape is: (batch_size, channels, length)
# For one sequence, batch size is 1, channels are 4, length is 7.  -> (1, 4, 7)

# For length 7 and kernel size 3, output length is: 7 - 3 + 1 = 5
# So output shape becomes: (1, 16, 5)
#first index the kernel can start = 1
#last index the kernel can start  = L − K + 1
# 1 2 3 4 5 6 7
# A A A U G C C

# Then global pooling
# To get one prediction for the whole sequence, we pool across the length dimension.
# For each of the 16 filters, keep the strongest activation anywhere in the sequence.
# That means:
# Did motif-like pattern #1 appear strongly?
# Did motif-like pattern #2 appear strongly?
# ...
# Did motif-like pattern #16 appear strongly?


In [ ]:
#  -------- toy example with multiple sequences and targets
sequences = [
    "AAAUGCC",
    "AUGCGAA",
    "UUUGGCA"
]
targets = [
    0.8,
    0.3,
    0.6
]
X = [one_hot_encode(seq) for seq in sequences]
Y = torch.tensor(targets, dtype=torch.float32)

# --- minimal CNN model ---
class SeqCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=4, 
                               out_channels=16, 
                               kernel_size=3 #each filter looks at a window of 3 positions
                               )
        self.fc = nn.Linear(16, 1)
    def forward(self, x):
        #print("forward pass")
        #print(x.shape)
        x = self.conv1(x) # 16 learned features per position
        #print("after conv layer")
        #print(x.shape)
        x = torch.relu(x) # This introduces non-linearity.
        #print("before mean pooling")
        #print(x.shape)
        #print(x)
        x = torch.max(x, dim=2).values # global max pooling across sequence length
        #print("after mean pooling")
        #print(x.shape)
        x = self.fc(x) # scalar prediction
        return x

In [ ]:
seq = "AAAUGCC"
x = one_hot_encode(seq)   # (7, 4)
x = x.T.unsqueeze(0)      # (1, 4, 7)
# x.T changes (7, 4) to (4, 7)
#unsqueeze(0) adds batch dimension: (1, 4, 7)
print(x)

tensor([[[1., 1., 1., 0., 0., 0., 0.],
         [0., 0., 0., 1., 0., 0., 0.],
         [0., 0., 0., 0., 0., 1., 1.],
         [0., 0., 0., 0., 1., 0., 0.]]])


In [ ]:
model = SeqCNN()
model.forward(x)

forward pass
torch.Size([1, 4, 7])
after conv layer
torch.Size([1, 16, 5])
before mean pooling
torch.Size([1, 16, 5])
tensor([[[0.1194, 0.0000, 0.1892, 0.1338, 0.2026],
         [0.2540, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0509, 0.2049, 0.1873],
         [0.0000, 0.0000, 0.1098, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1565, 0.2316, 0.5229, 0.5000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1660, 0.0980, 0.6851, 0.5256, 0.4642],
         [0.3622, 0.1589, 0.0000, 0.0000, 0.1897],
         [0.4228, 0.3924, 0.1380, 0.0782, 0.2953],
         [0.1241, 0.2094, 0.2332, 0.0886, 0.0613],
         [0.0000, 0.0000, 0.4071, 0.1587, 0.0791],
         [0.0345, 0.0000, 0.0676, 0.7484, 0.1079],
         [0.2284, 0.7349, 0.4063, 0.4188, 0.0000],
         [0.1143, 0.1761, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0879, 0.0000, 0.6848, 0.3967]]], grad_fn=<ReluBackward0>)
after mean pooling
torch.Size([1, 16])


tensor([[-0.0268]], grad_fn=<AddmmBackward0>)

In [ ]:
#  -------- toy example with multiple sequences and targets
sequences = [
    "AAAUGCC",
    "AUGCGAA",
    "UUUGGCA"
]
targets = [
    0.8,
    0.3,
    0.6
]
X = [one_hot_encode(seq).T.unsqueeze(0) for seq in sequences]
#print(X)
Y = torch.tensor(targets, dtype=torch.float32)

# Since all sequences have the same length, you can stack them:
#X = torch.stack([one_hot_encode(seq).T for seq in sequences])
#Y = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)
#  -------- run model
model = SeqCNN()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -------- training loop --------
for epoch in range(10): # one pass through the entire dataset
    total_loss = 0.0
    for x, y in zip(X,Y):
        optimizer.zero_grad()
        prediction = model(x) # forward pass to get the prediction
        #print(prediction)
        #print(prediction.squeeze())
        loss = loss_fn(prediction.squeeze(), y) # Measure how wrong the prediction is.
        loss.backward() # Compute gradients for all weights.
        optimizer.step() #  Optimizer updates the weights using those gradients.
        total_loss += loss.item()
    avg_loss = total_loss / len(X)
    print(f"epoch={epoch}, avg_loss={avg_loss:.2f}")

epoch=0, avg_loss=0.33
epoch=1, avg_loss=0.29
epoch=2, avg_loss=0.26
epoch=3, avg_loss=0.23
epoch=4, avg_loss=0.21
epoch=5, avg_loss=0.18
epoch=6, avg_loss=0.16
epoch=7, avg_loss=0.14
epoch=8, avg_loss=0.13
epoch=9, avg_loss=0.11


In [ ]:
seq = "AAAUGCC"
x_new = one_hot_encode(seq)   # (7, 4)
x_new = x_new.T.unsqueeze(0)      # (1, 4, 7)

# Switch model to evaluation mode
model.eval()
# Disable gradients and run the model
with torch.no_grad(): # disables gradient tracking
    prediction = model(x_new)
#print(prediction)
score = prediction.item()
print(f"Predicted score: {score:.2f}")

tensor([[0.3257]])
Predicted score: 0.33


In [71]:
def predict_sequence(seq, model):
    x = one_hot_encode(seq).T.unsqueeze(0)
    model.eval()
    with torch.no_grad():
        pred = model(x)
    return pred.item()

seq = "AAAUGCC"
score = predict_sequence(seq, model)
print(f"Score: {score:.2f}")

Score: 0.33


In [ ]:
# Dropout is a regularization technique to prevent overfitting.
# During training it randomly turns off neurons.
# This forces the network to not rely too much on any single neuron.
# before dropout
# [0.3 0.7 0.5 0.2]
# after dropout
# [0.3 0.0 0.5 0.0]
# During prediction we do NOT want randomness.
# So when you call model.eval() dropout is automatically disabled.

# BatchNorm stabilizes training by normalizing activations.
# During training it computes statistics from the current batch: mean, variance
# During prediction there is no batch distribution to estimate.
# So BatchNorm instead uses running averages learned during training.
# Again, model.eval() switches it to this mode.


In [72]:
class SeqCNN_batchnorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(4, 16, kernel_size=3)
        self.bn1 = nn.BatchNorm1d(16) # normalize each of the 16 channels
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = torch.max(x, dim=2).values
        x = self.fc(x)
        return x

In [73]:
class SeqCNN_dropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(4, 16, kernel_size=3)
        self.dropout = nn.Dropout(p=0.3)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = torch.relu(x)
        x = torch.max(x, dim=2).values
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [ ]:
# In a CNN, batch normalization is commonly 
# added after the convolution and before the
# activation, while dropout is often added 
# after pooling or before the final dense 
# layer to reduce overfitting.


In [77]:
def train_model(model_arch,
                X,
                Y,
                lr=0.001,
                n_epochs=10):
    model = model_arch()
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(n_epochs): # one pass through the entire dataset
        total_loss = 0.0
        for x, y in zip(X,Y):
            optimizer.zero_grad()
            prediction = model(x) # forward pass to get the prediction
            #print(prediction)
            #print(prediction.squeeze())
            loss = loss_fn(prediction.squeeze(), y) # Measure how wrong the prediction is.
            loss.backward() # Compute gradients for all weights.
            optimizer.step() #  Optimizer updates the weights using those gradients.
            total_loss += loss.item()
        avg_loss = total_loss / len(X)
        print(f"epoch={epoch}, avg_loss={avg_loss:.2f}")
    return(model)

print("------------SeqCNN")
m_seqcnn = train_model(SeqCNN, X, Y)
print("------------SeqCNN_batchnorm")
m_seqcnn_batchnorm = train_model(SeqCNN_batchnorm, X, Y)
print("------------SeqCNN_dropout")
m_seqcnn_dropout = train_model(SeqCNN_dropout, X, Y)  

------------SeqCNN
epoch=0, avg_loss=0.13
epoch=1, avg_loss=0.11
epoch=2, avg_loss=0.09
epoch=3, avg_loss=0.07
epoch=4, avg_loss=0.06
epoch=5, avg_loss=0.05
epoch=6, avg_loss=0.04
epoch=7, avg_loss=0.03
epoch=8, avg_loss=0.03
epoch=9, avg_loss=0.02
------------SeqCNN_batchnorm
epoch=0, avg_loss=1.92
epoch=1, avg_loss=1.63
epoch=2, avg_loss=1.37
epoch=3, avg_loss=1.15
epoch=4, avg_loss=0.95
epoch=5, avg_loss=0.77
epoch=6, avg_loss=0.62
epoch=7, avg_loss=0.50
epoch=8, avg_loss=0.39
epoch=9, avg_loss=0.31
------------SeqCNN_dropout
epoch=0, avg_loss=0.24
epoch=1, avg_loss=0.23
epoch=2, avg_loss=0.32
epoch=3, avg_loss=0.15
epoch=4, avg_loss=0.12
epoch=5, avg_loss=0.29
epoch=6, avg_loss=0.10
epoch=7, avg_loss=0.13
epoch=8, avg_loss=0.24
epoch=9, avg_loss=0.32


In [ ]:
# BatchNorm is usually helpful with real batches
# your training loop uses one sequence at a time
# so batch size is effectively 1

# dropout randomly removes features during training
# with only 3 examples, that randomness is huge relative to dataset size
# regularization makes optimization harder
# on such a tiny dataset, dropout can hurt more than help
# IF I'd like to analyse more batches at one (more than one seqneucen at once then)
# I shoudl stakc them and then train
X_batch = torch.stack([one_hot_encode(seq).T for seq in sequences])
Y_batch = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)
# ...
for epoch in range(10):
    optimizer.zero_grad()
    prediction = model(X_batch)
    loss = loss_fn(prediction, Y_batch)
    loss.backward()
    optimizer.step()
    print(f"epoch={epoch}, loss={loss.item():.3f}")


In [ ]:
# Or even better: Dataloader:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class RNADataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        y = self.targets[idx]
        x = one_hot_encode(seq).T   # (4, seq_len)
        return x, torch.tensor(y, dtype=torch.float32)
    
dataset = RNADataset(sequences, targets)
loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)
#### Training using Dataloader
model = SeqCNN()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(10):
    model.train()
    total_loss = 0
    for x_batch, y_batch in loader:
        optimizer.zero_grad()
        prediction = model(x_batch)
        loss = loss_fn(prediction.squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    print(f"epoch={epoch}, avg_loss={avg_loss:.3f}")

In [ ]:
# Use nn.Sequential to simplify model definition when 
# layers are in a simple sequence without branching.:
class RNASeqCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(4, 16, kernel_size=5),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
            nn.Flatten(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

If it's a classificaiton task:

In [ ]:
# If it's a classification task then use nn.BCEWithLogitsLoss()
#During prediction / inference
#Then you apply sigmoid yourself to convert logits into probabilities:
class RNADataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        x = one_hot_encode(self.sequences[idx])              # (4, L)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y

class RNASeqCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(4, 16, kernel_size=5),
            nn.ReLU(),
            nn.BatchNorm1d(16),
            nn.Conv1d(16, 32, kernel_size=3),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),  # (batch, 32, 1)
            nn.Flatten(),             # (batch, 32)
            nn.Linear(32, 1)          # logits
        )
    def forward(self, x):
        return self.net(x).squeeze(1)  # (batch,)
sequences = [
    "AUGCUAACGU",
    "CCCCGAUUUA",
    "GGGAUACGUA",
    "UUUAAACCCG",
    "ACGUACGUAC",
    "GGGGUUUUAA"
]
labels = [1, 0, 1, 0, 1, 0]
dataset = RNADataset(sequences, labels)
loader = DataLoader(dataset, batch_size=2, shuffle=True)


# Training
model = RNASeqCNN()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop 
for epoch in range(5):
    model.train()
    total_loss = 0.0
    for x_batch, y_batch in loader:
        optimizer.zero_grad()
        logits = model(x_batch)                     # (batch,)
        loss = criterion(logits, y_batch)           # y: float 0/1
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, loss={total_loss:.4f}")

# Prediciton
model.eval()
with torch.no_grad():
    test_seq = one_hot_encode("AUGGCUACGU").unsqueeze(0)  # (1, 4, L)
    logits = model(test_seq)
    probs = torch.sigmoid(logits)
    pred = (probs >= 0.5).int()
    print("Probability:", probs.item())
    print("Prediction:", pred.item())

In [ ]:
# transformer
# But unlike your MLP, and even unlike a CNN, 
# a Transformer can model relationships between positions across the whole sequence.

#Transformer encoder
#Learns contextual representations:
#what is at this position
#what is around it
#what other positions matter for it
#So it is good when:
#- long-range dependencies matter
#- position interactions matter
#- motif combinations matter


# For RNA:
# A A A U G C C
# A Transformer encoder lets each position “look at” the others using self-attention.#
# So position 4 (U) can attend to:
# nearby positions
# distant positions
# whatever the model learns is important

# Simplest version ever:


#sequence
#↓
#token encoding / embedding
#↓
#positional encoding
#↓
#TransformerEncoder
#↓
#pool across sequence
#↓
#linear layer
#↓
#scalar prediction


# Input:
# 1. integer tokens
# A → 0
# U → 1
# G → 2
# C → 3
# Then sequence "AAAUGCC" becomes:
# [0, 0, 0, 1, 2, 3, 3]


# 1. Tokenize
# AAAUGCC
# ↓
# [0, 0, 0, 1, 2, 3, 3]

# 2. Embedding layer
# self.embedding = nn.Embedding(num_embeddings=4, embedding_dim=d_model)
# This turns each token into a 16-dimensional vector.
# (1, 7) → (1, 7, 16)
# 
# 3. Positional encoding
# Transformers do not automatically know sequence order, so we add positional information.
# This can be:
# learned positional embeddings
# sinusoidalpositional encoding

# 4. Transformer encoder
# It outputs contextualized representations for each position.

# 5. Pooling
# To get one prediction per sequence, we reduce across sequence length.
# (1, 7, 16) → (1, 16)

# 6. Final linear layer
# self.fc = nn.Linear(d_model, 1)
# Shape:
# (1, 16) → (1, 1)


##########
#tokens:             (1, 7)
#embedding:          (1, 7, 16)
#+ positional emb:   (1, 7, 16)
#transformer:        (1, 7, 16)
#mean pool:          (1, 16)
#linear:             (1, 1)

# I’d use a CNN when I mainly care about local motifs, 
# and a Transformer encoder when I expect longer-range 
# dependencies or more complex interactions between positions.

# Query: what patterns do I want to find?
# Key: what information do I contain?
# Value: what information should be passed along?
# attention_score = Q · Kᵀ

# Transformers usually run several attention mechanisms in parallel.
# Example:
# 4 heads
# head 1 → motif detection
# head 2 → GC content patterns
# head 3 → long-range dependencies
# head 4 → positional effects

#embedding         (1, 7, 16)
#Q, K, V           (1, 7, 16)
#attention matrix  (1, 7, 7)
#output vectors    (1, 7, 16)

#In self-attention, each token is projected into query, key, and value vectors.
#Attention scores are computed between queries and keys to determine how
#much each position attends to others. The values are then combined
#using these attention weights to produce contextualized representations.

# Attention(Q, K, V) = softmax( QKᵀ / √d ) V
#tokens
# ↓
#embedding
# ↓
#Q, K, V projections
# ↓
#QKᵀ similarity matrix
# ↓
#softmax attention weights  -> This turns each row into weights that sum to 1.
# ↓
#weighted combination of V
# ↓
#context-aware token representations


# Intuitive explanation
# Think of it like this:
# Each nucleotide asks:
# Which other nucleotides are relevant to me?
# The attention weights answer that question.


#Regression	nn.MSELoss
#Binary class	nn.BCEWithLogitsLoss
#Multi-class	nn.CrossEntropyLoss


In [ ]:
class SeqTransformer(nn.Module):
    def __init__(self, 
                 d_model=16, 
                 nhead=4, 
                 num_layers=2, 
                 max_len=100):
        super().__init__()

        self.embedding = nn.Embedding(4, d_model)         # A,U,G,C -> vectors
        self.pos_embedding = nn.Embedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=64,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        # x shape: (batch, seq_len), integer tokens
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        x = self.embedding(x) + self.pos_embedding(positions)  # (batch, seq_len, d_model)
        x = self.transformer(x)                                # (batch, seq_len, d_model)
        x = x.mean(dim=1)                                      # (batch, d_model)
        x = self.fc(x)                                         # (batch, 1)
        return x

def encode_seq_as_tokens(seq):
    mapping = {"A": 0, "U": 1, "G": 2, "C": 3}
    return torch.tensor([mapping[n] for n in seq], dtype=torch.long)



In [ ]:
seq = "AAAUGCC"
x = encode_seq_as_tokens(seq).unsqueeze(0)   # (1, 7)

model = SeqTransformer()
out = model(x)
score = out.item()
print(f"Score: {score:.2f}")

Score: -0.86


In [87]:
sequences = [
    "AAAUGCC",
    "AUGCGAA",
    "UUUGGCA"
]

targets = [0.8, 0.3, 0.6]

X = torch.stack([encode_seq_as_tokens(seq) for seq in sequences])   # (3, 7)
Y = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)         # (3, 1)

model = SeqTransformer()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()

    optimizer.zero_grad()
    prediction = model(X)
    loss = loss_fn(prediction, Y)
    loss.backward()
    optimizer.step()

    print(f"epoch={epoch}, loss={loss.item():.3f}")

epoch=0, loss=0.790
epoch=1, loss=0.630
epoch=2, loss=0.509
epoch=3, loss=0.334
epoch=4, loss=0.282
epoch=5, loss=0.200
epoch=6, loss=0.156
epoch=7, loss=0.132
epoch=8, loss=0.101
epoch=9, loss=0.096


# Variational autoencoder

input x
↓
encoder
↓
mu, logvar
↓
sample z
↓
decoder
↓
reconstructed x_hat

Here the loss here is: "reconstruction loss + KL divergence"


Unlike a normal autoencoder, a VAE does not map each input to one fixed latent point. It learns a distribution in latent space: z ~ N(mu, sigma^2).


A VAE has an encoder that outputs the mean and log-variance of a latent Gaussian distribution, a reparameterization step to sample a latent vector, and a decoder that reconstructs the input. The training objective combines reconstruction loss with a KL divergence term that regularizes the latent space toward a standard normal distribution.
x → encoder → (mu, logvar) → sample z → decoder → x_hat

In [91]:
# Minimal VAE in PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

class SequenceVAE(nn.Module):
    def __init__(self, seq_len=7, alphabet_size=4, hidden_dim=64, latent_dim=16):
        super().__init__()
        input_dim = seq_len * alphabet_size

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        self.fc2 = nn.Linear(latent_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)

        self.seq_len = seq_len
        self.alphabet_size = alphabet_size

    def encode(self, x):
        # x shape: (batch, seq_len, 4)
        x = x.view(x.size(0), -1)
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc2(z))
        x_hat = torch.sigmoid(self.fc3(h))
        x_hat = x_hat.view(-1, self.seq_len, self.alphabet_size)
        return x_hat

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

In [ ]:
# Training loop for VAE
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# ---- toy data ----
sequences = [
    "AAAUGCC",
    "AUGCGAA",
    "UUUGGCA"
]
targets = [0.8, 0.3, 0.6]

# ---- create dataset ----
class RNADataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        y = self.targets[idx]
        x = one_hot_encode(seq).T   # (4, seq_len)
        return x, torch.tensor(y, dtype=torch.float32)
dataset = RNADataset(sequences, targets)

# ---- create dataloader ----
loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

# ---- create dataloader ----

model = SequenceVAE(seq_len=7, alphabet_size=4, hidden_dim=64, latent_dim=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
def vae_loss(x_hat, x, mu, logvar):
    recon_loss = F.binary_cross_entropy(x_hat, x, reduction="sum")
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

for epoch in range(50):
    model.train()
    total_loss = 0

    for x_batch, y_batch in loader:
        optimizer.zero_grad()

        x_hat, mu, logvar = model(x_batch)
        loss = vae_loss(x_hat, # x_hat -> [2, 7, 4]
                        x_batch.transpose(1, 2), # x_batch -> [2, 4, 7]
                        mu, 
                        logvar)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"epoch={epoch}, loss={total_loss:.2f}")

epoch=0, loss=61.05
epoch=1, loss=61.75
epoch=2, loss=60.19
epoch=3, loss=60.73
epoch=4, loss=59.88
epoch=5, loss=60.39
epoch=6, loss=60.27
epoch=7, loss=58.10
epoch=8, loss=60.70
epoch=9, loss=60.27
epoch=10, loss=59.16
epoch=11, loss=60.32
epoch=12, loss=60.94
epoch=13, loss=59.48
epoch=14, loss=59.13
epoch=15, loss=59.39
epoch=16, loss=60.14
epoch=17, loss=59.57
epoch=18, loss=59.71
epoch=19, loss=58.89
epoch=20, loss=59.18
epoch=21, loss=59.82
epoch=22, loss=58.91
epoch=23, loss=58.02
epoch=24, loss=57.85
epoch=25, loss=59.91
epoch=26, loss=58.18
epoch=27, loss=58.09
epoch=28, loss=59.31
epoch=29, loss=58.95
epoch=30, loss=57.88
epoch=31, loss=58.08
epoch=32, loss=58.84
epoch=33, loss=58.02
epoch=34, loss=57.24
epoch=35, loss=58.59
epoch=36, loss=57.30
epoch=37, loss=59.22
epoch=38, loss=57.16
epoch=39, loss=57.25
epoch=40, loss=58.16
epoch=41, loss=58.72
epoch=42, loss=57.87
epoch=43, loss=56.89
epoch=44, loss=56.73
epoch=45, loss=57.17
epoch=46, loss=56.69
epoch=47, loss=57.36
ep

In [ ]:
# MLP/CNN/Transformer: sequence → predict a value
# VAE: sequence → (reconstructed sequence, latent representation); reconstruct + embed + generate

# A trained VAE can be used to reconstruct sequences, 
# embed sequences into a latent space via the encoder, 
# or generate new sequences by sampling from the latent space and decoding.

# VAE:
## Option 1: reconstruct a sequence
def reconstruct_sequence(seq, model):
    x = one_hot_encode(seq).unsqueeze(0)  # (1, seq_len, 4)
    model.eval()
    with torch.no_grad():
        x_hat, mu, logvar = model(x)
    return x_hat
seq = "AAAUGCC"
x_hat = reconstruct_sequence(seq, model)
print(x_hat)


# Make it human-readable (decode back to letters)
# Convert probabilities → nucleotide:
def decode_one_hot(x_hat):
    idx_to_nt = ["A", "U", "G", "C"]
    indices = x_hat.argmax(dim=-1)  # (batch, seq_len)
    seqs = []
    for row in indices:
        seq = "".join([idx_to_nt[i] for i in row])
        seqs.append(seq)
    return seqs
decoded = decode_one_hot(x_hat)
print(decoded)

tensor([[[0.5471, 0.5516, 0.4745, 0.5045],
         [0.4900, 0.5124, 0.4788, 0.3979],
         [0.5037, 0.4836, 0.4849, 0.5050],
         [0.4709, 0.4929, 0.3870, 0.5238],
         [0.4822, 0.4669, 0.4710, 0.4713],
         [0.3846, 0.4889, 0.4933, 0.4637],
         [0.5907, 0.4816, 0.4966, 0.4259]]])
['UUCCAGA']


In [98]:
## Option 2: get latent representation
def encode_sequence(seq, model):
    x = one_hot_encode(seq).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        mu, logvar = model.encode(x)

    return mu
z = encode_sequence("AAAUGCC", model)
print(z)


tensor([[ 0.1458, -0.0539,  0.0041, -0.2022, -0.0864, -0.0671, -0.0117, -0.2428,
          0.0477, -0.0415,  0.0099, -0.0673,  0.0429, -0.1086,  0.0803, -0.0331]])


In [ ]:
## Option 3: generate new sequences
def generate_sequence(model, latent_dim):
    z = torch.randn(1, latent_dim)
    model.eval()
    with torch.no_grad():
        x_hat = model.decode(z)
    return decode_one_hot(x_hat)
new_seq = generate_sequence(model, latent_dim=16)
print(new_seq)

['UCAUAGA']
